### SQLite에 대화 내용 저장하기

- SQL : Structured Query Language 데이터베이스에서 데이터를 저장,조회,수정,삭제하는데 사용하는 언어
- SQLite : SQL 명령어를 잉요해 데이터를 저장하고 관리하는 가벼운 데이터베이스 관리 시스템

* storage를 사용하려면 session_id, connection을 제공해야 합니다

In [1]:
!pip --version

pip 26.2.1 from d:\hanhwa0902\ex0918\.0918venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

In [5]:
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 클리드야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)

chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")

In [6]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 클리드야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [7]:
from langchain_core.prompts import(
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

chain = prompt | ChatOpenAI(model="gpt-5-mini") | StrOutputParser()

In [9]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db",
    )

In [10]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

In [11]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
    history_factory_config=config_fields,
)

d:\hanhwa0902\ex0918\.0918venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [13]:
chain_with_history.invoke({"question": "안녕 반가워, 내 이름은 테디야"}, config)

'안녕 테디! 만나서 반가워. 나는 AI 어시스턴트야 — 뭐 도와줄까? 필요한 게 있으면 편하게 말해줘.'

In [14]:
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'네 이름은 테디야.'

In [15]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'지금 이 대화에서는 당신의 이름을 알 수 없어요. 알려주시면 앞으로 그 이름으로 불러드릴게요. 알려주기 싫으시면 별칭(닉네임)이나 호칭 방식(예: 성함+님, 그냥 이름 등)도 알려주세요.'